# Solstice Opal Ancillary Revenue Analysis

This notebook analyzes which hotel guest segments generate the most ancillary revenue from dining, spa, and activities.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

The analysis uses three CSV files: guest profiles, stay details, and ancillary spend transactions.

In [ ]:
guest_profiles = pd.read_csv('da_sample_guest_profiles.csv')
stay_details = pd.read_csv('da_sample_stay_details.csv')
spend = pd.read_csv('da_sample_ancillary_spend.csv')

print('guest_profiles:', guest_profiles.shape)
print('stay_details:', stay_details.shape)
print('ancillary_spend:', spend.shape)

## 3. Data Validation

Validation checks found missing loyalty tiers, duplicate stay IDs, letter suffixes in ancillary spend guest IDs, and missing spend amounts.

In [ ]:
print('Missing loyalty tiers:', guest_profiles['loyalty_tier'].isna().sum())
print('Duplicate stay IDs:', stay_details['stay_id'].duplicated().sum())
print('Missing spend amounts:', spend['amount_spent'].isna().sum())
print('Spend categories:')
print(spend['category'].value_counts())

## 4. Data Cleaning

Cleaning steps: remove the letter suffix from spend guest IDs, drop missing spend amounts, remove duplicate stay IDs, and fill missing loyalty tiers as Non-member.

In [ ]:
spend_clean = spend.copy()
spend_clean['guest_id'] = spend_clean['guest_id'].astype(str).str.extract(r'(\d+)').astype(int)
spend_clean = spend_clean.dropna(subset=['amount_spent'])

stays_clean = stay_details.drop_duplicates(subset=['stay_id'], keep='first').copy()
stays_clean['check_in_date'] = pd.to_datetime(stays_clean['check_in_date'])

guests_clean = guest_profiles.copy()
guests_clean['loyalty_tier'] = guests_clean['loyalty_tier'].fillna('Non-member')

## 5. Merge Tables

The cleaned tables are merged into one analysis table, with ancillary spend aggregated by guest.

In [ ]:
merged = stays_clean.merge(guests_clean, on='guest_id', how='left')

spend_agg = spend_clean.groupby('guest_id').agg(
    total_ancillary_spend=('amount_spent', 'sum'),
    num_transactions=('amount_spent', 'count')
).reset_index()

full = merged.merge(spend_agg, on='guest_id', how='left')
full['total_ancillary_spend'] = full['total_ancillary_spend'].fillna(0)
full['num_transactions'] = full['num_transactions'].fillna(0)

full.head()

## 6. Exploratory Analysis

In [ ]:
total_revenue = full['total_ancillary_spend'].sum()
total_stays = len(full)
avg_per_stay = total_revenue / total_stays
zero_spend = (full['total_ancillary_spend'] == 0).sum()

print(f'Total ancillary revenue: ${total_revenue:,.2f}')
print(f'Total stays: {total_stays}')
print(f'Average revenue per stay: ${avg_per_stay:.2f}')
print(f'Zero-spend stays: {zero_spend} ({100 * zero_spend / total_stays:.1f}%)')

In [ ]:
tier_order = ['Non-member', 'Silver', 'Gold', 'Platinum']

tier_analysis = full.groupby('loyalty_tier')['total_ancillary_spend'].agg(
    avg_spend='mean',
    total_spend='sum',
    num_stays='count'
).reindex(tier_order)

tier_analysis.round(2)

In [ ]:
reason_analysis = full.groupby('reason_for_stay')['total_ancillary_spend'].agg(
    avg_spend='mean',
    total_spend='sum',
    num_stays='count'
)

reason_analysis.round(2)

In [ ]:
channel_analysis = full.groupby('booking_channel')['total_ancillary_spend'].agg(
    avg_spend='mean',
    total_spend='sum',
    num_stays='count'
).sort_values('avg_spend', ascending=False)

channel_analysis.round(2)

In [ ]:
category_spend = spend_clean.groupby('category')['amount_spent'].sum().sort_values(ascending=False)
category_spend.round(2)

## 7. Key Findings

- Total ancillary revenue was $101,898.13 across 571 stays.
- Average ancillary revenue per stay was $178.46.
- Platinum guests had the highest average spend at $351.86 per stay.
- Leisure guests spent slightly more than business guests on average.
- Website bookings had the highest average ancillary spend by booking channel.
- Dining generated the most ancillary revenue by category.